# TRM Training - Corrected Implementation (Paper Algorithm 3)

**Based on:** *Less is More: Recursive Reasoning with Tiny Networks*  
**Target**: 70-87% puzzle accuracy with ~5M parameters  
**Dataset**: Kaggle 3M Sudoku  

**Key Improvements:**
- ✓ 2-layer tiny network (not 4!)
- ✓ Proper T-cycle warmup (T-1 no-grad + 1 grad)
- ✓ EMA for stability
- ✓ Deep supervision (N_sup=16)
- ✓ Only supervise final output (not intermediate z)

## 1. Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Install dependencies
!pip install torch numpy pandas -q

## 2. Model Definition (2 Layers!)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import LambdaLR
import numpy as np
import time
from typing import Tuple, Optional

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# RMSNorm
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor):
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return x / rms * self.weight


# RoPE (Fixed)
class RotaryPositionalEncoding(nn.Module):
    def __init__(self, dim: int, max_seq_len: int = 2048, base: float = 10000.0):
        super().__init__()
        self.dim = dim
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len: int):
        t = torch.arange(seq_len, device=self.inv_freq.device)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def _rotate_half(self, x):
        x1, x2 = x.chunk(2, dim=-1)
        return torch.cat([-x2, x1], dim=-1)

    def forward(self, q: torch.Tensor, k: torch.Tensor):
        seq_len = q.size(2)
        cos = self.cos_cached[:seq_len, :].unsqueeze(0).unsqueeze(0)
        sin = self.sin_cached[:seq_len, :].unsqueeze(0).unsqueeze(0)
        q_rot = q * cos + self._rotate_half(q) * sin
        k_rot = k * cos + self._rotate_half(k) * sin
        return q_rot, k_rot


# SwiGLU
class SwiGLU(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: Optional[int] = None):
        super().__init__()
        hidden_dim = hidden_dim or int(2 / 3 * 4 * in_dim)
        hidden_dim = ((hidden_dim + 63) // 64) * 64
        self.w1 = nn.Linear(in_dim, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, in_dim, bias=False)
        self.w3 = nn.Linear(in_dim, hidden_dim, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


# Attention
class TinyAttention(nn.Module):
    def __init__(self, dim: int, n_heads: int = 8, dropout: float = 0.0, max_seq_len: int = 2048):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(dim, 3 * dim, bias=False)
        self.out = nn.Linear(dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.rope = RotaryPositionalEncoding(self.head_dim, max_seq_len)

    def forward(self, x, mask=None):
        B, L, D = x.shape
        qkv = self.qkv(x).reshape(B, L, 3, self.n_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        q, k = self.rope(q, k)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        if mask is not None:
            attn = attn.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)
        out = (attn @ v).transpose(1, 2).reshape(B, L, D)
        return self.out(out)


# Block
class TinyBlock(nn.Module):
    def __init__(self, dim: int, n_heads: int = 8, dropout: float = 0.0, max_seq_len: int = 2048):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.attn = TinyAttention(dim, n_heads, dropout, max_seq_len)
        self.norm2 = RMSNorm(dim)
        self.ffn = SwiGLU(dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        x = x + self.dropout(self.attn(self.norm1(x), mask))
        x = x + self.dropout(self.ffn(self.norm2(x)))
        return x


# TinyNet (2 LAYERS!)
class TinyNet(nn.Module):
    def __init__(self, dim=512, n_layers=2, n_heads=8, max_seq_len=1024, dropout=0.0):
        super().__init__()
        self.dim = dim
        self.layers = nn.ModuleList([TinyBlock(dim, n_heads, dropout, max_seq_len) for _ in range(n_layers)])
        self.norm = RMSNorm(dim)

    def forward(self, x, mask=None):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

In [ ]:
# Heads
class CombinedHead(nn.Module):
    def __init__(self, dim: int, vocab_size: int):
        super().__init__()
        self.output_head = nn.Linear(dim, vocab_size, bias=False)
        self.halt_head = nn.Sequential(
            nn.Linear(dim, dim // 4),
            nn.ReLU(),
            nn.Linear(dim // 4, 1)
        )

    def forward(self, y):
        y_hat = self.output_head(y)
        q_hat = self.halt_head(y.mean(dim=1))
        return y_hat, q_hat


# Latent Recursion (Algorithm 3, lines 413-417)
class LatentRecursion(nn.Module):
    def __init__(self, net: TinyNet, dim: int = 512, n_latent: int = 6):
        super().__init__()
        self.net = net
        self.n_latent = n_latent
        self.z_proj = nn.Linear(3 * dim, dim, bias=False)
        self.y_proj = nn.Linear(2 * dim, dim, bias=False)

    def forward(self, x, y, z):
        # Update z for n_latent steps
        for _ in range(self.n_latent):
            concat = torch.cat([x, y, z], dim=-1)
            z = self.net(self.z_proj(concat))
        
        # Update y once
        concat_y = torch.cat([y, z], dim=-1)
        y_new = self.net(self.y_proj(concat_y))
        return y_new, z


# TRM Model with T-cycle warmup (Algorithm 3, lines 418-425)
class TRMModel(nn.Module):
    def __init__(self, dim=512, n_layers=2, n_heads=8, n_latent=6, T_cycles=3, vocab_size=10, max_seq_len=128, dropout=0.0):
        super().__init__()
        self.dim = dim
        self.T_cycles = T_cycles
        self.net = TinyNet(dim, n_layers, n_heads, max_seq_len, dropout)
        self.latent_recursion = LatentRecursion(self.net, dim, n_latent)
        self.heads = CombinedHead(dim, vocab_size)

    def forward(self, x, y, z):
        # T-1 cycles WITHOUT gradients (warmup)
        with torch.no_grad():
            for _ in range(self.T_cycles - 1):
                y, z = self.latent_recursion(x, y, z)
        
        # Final cycle WITH gradients
        y, z = self.latent_recursion(x, y, z)
        y_hat, q_hat = self.heads(y)
        return (y, z), y_hat, q_hat

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

## 3. Load Dataset

In [ ]:
import pandas as pd
import os

print("Loading Sudoku dataset from Kaggle...")
print("Dataset: 3 Million Sudoku Puzzles with Ratings")
print("="*60)

# Set Kaggle credentials
os.environ['KAGGLE_USERNAME'] = 'aayushbhure'  # CHANGE THIS!
os.environ['KAGGLE_KEY'] = 'KGAT_4bb3a6797b5560ffe3b19de1e8035651'

# Install Kaggle
!pip install -q kaggle

# Download dataset
print("Downloading dataset from Kaggle...")
!kaggle datasets download -d radcliffe/3-million-sudoku-puzzles-with-ratings
!unzip -q 3-million-sudoku-puzzles-with-ratings.zip

print("✓ Dataset downloaded")

# Load CSV
df = pd.read_csv('sudoku-3m.csv')
print(f"\nTotal puzzles: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")

# Convert to our format
train_data = []
N_PUZZLES = 50000

for idx, row in df.head(N_PUZZLES).iterrows():
    try:
        puzzle_str = str(row['puzzle'])
        solution_str = str(row['solution'])
        
        puzzle = []
        for c in puzzle_str:
            if c.isdigit():
                puzzle.append(int(c))
            elif c in ['.', '0']:
                puzzle.append(0)
        
        solution = [int(c) for c in solution_str if c.isdigit()]
        
        if len(puzzle) == 81 and len(solution) == 81:
            train_data.append({
                "puzzle": puzzle,
                "solution": solution
            })
        
        if (idx + 1) % 10000 == 0:
            print(f"  Processed {idx + 1:,} puzzles...")
    except:
        continue

print("="*60)
print(f"✓ Loaded {len(train_data):,} Sudoku puzzles!")

# Show sample
sample = train_data[0]
print(f"\nSample puzzle (0 = blank):")
puzzle_grid = np.array(sample['puzzle']).reshape(9, 9)
print(puzzle_grid)

In [ ]:
# Dataset
class SudokuDataset(Dataset):
    def __init__(self, data, max_seq_len=128):
        self.data = data
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        puzzle = torch.tensor(item["puzzle"], dtype=torch.long)
        solution = torch.tensor(item["solution"], dtype=torch.long)
        
        x_tokens = torch.zeros(self.max_seq_len, dtype=torch.long)
        x_tokens[:81] = puzzle
        
        y_init = puzzle.clone()
        blank_mask = puzzle == 0
        y_init[blank_mask] = torch.randint(1, 10, (blank_mask.sum(),))
        y_init_tokens = torch.zeros(self.max_seq_len, dtype=torch.long)
        y_init_tokens[:81] = y_init
        
        y_true = torch.zeros(self.max_seq_len, dtype=torch.long)
        y_true[:81] = solution
        
        return {"x_tokens": x_tokens, "y_init_tokens": y_init_tokens, "y_true": y_true}

## 4. Training Configuration

In [ ]:
# ============================================================
# CORRECTED CONFIG (Following Paper)
# ============================================================

CONFIG = {
    # Model Architecture
    "dim": 512,
    "n_layers": 2,           # ✓ 2 layers (paper recommendation)
    "n_heads": 8,
    "n_latent": 6,           # ✓ n=6 latent updates
    "T_cycles": 3,           # ✓ T=3 (2 warmup + 1 grad)
    "vocab_size": 10,
    "max_seq_len": 128,
    "dropout": 0.1,
    
    # Training
    "lr": 3e-4,
    "weight_decay": 0.01,
    "max_iters": 20000,
    "batch_size": 32,
    "warmup_steps": 1000,
    "grad_clip": 1.0,
    
    # Deep Supervision
    "N_sup": 16,             # ✓ Max supervision steps per sample
    "ema_decay": 0.999,      # ✓ EMA for stability
    
    # Checkpointing
    "save_every": 2000,
    "log_every": 100,
}

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"Expected training time on A100: ~60-90 minutes")
print(f"Expected accuracy: 70-87% (paper: 87.4%)")

In [ ]:
# Create dataloader
dataset = SudokuDataset(train_data)
dataloader = DataLoader(
    dataset, 
    batch_size=CONFIG["batch_size"],
    shuffle=True, 
    num_workers=2
)
print(f"Batches per epoch: {len(dataloader)}")
print(f"Total training samples: {len(dataset):,}")

In [ ]:
# Initialize model (2 LAYERS!)
model = TRMModel(
    dim=CONFIG["dim"],
    n_layers=2,  # ✓ ONLY 2 LAYERS!
    n_heads=CONFIG["n_heads"],
    n_latent=CONFIG["n_latent"],
    T_cycles=CONFIG["T_cycles"],
    vocab_size=CONFIG["vocab_size"],
    max_seq_len=CONFIG["max_seq_len"],
    dropout=CONFIG["dropout"]
).to(device)

embedding = nn.Embedding(20, CONFIG["dim"]).to(device)

optimizer = optim.AdamW(
    list(model.parameters()) + list(embedding.parameters()),
    lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"]
)

def lr_lambda(step):
    if step < CONFIG["warmup_steps"]:
        return step / CONFIG["warmup_steps"]
    return 1.0

scheduler = LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler('cuda')

print(f"Model parameters: {model.count_parameters():,}")
print(f"Embedding parameters: {sum(p.numel() for p in embedding.parameters()):,}")
print(f"Total parameters: {model.count_parameters() + sum(p.numel() for p in embedding.parameters()):,}")

In [ ]:
# EMA (Exponential Moving Average for stability)
class EMA:
    def __init__(self, model, decay=0.999):
        self.model = model
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()
    
    def update(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = self.decay * self.shadow[name] + (1 - self.decay) * param.data
    
    def apply_shadow(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data.clone()
                param.data = self.shadow[name]
    
    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                param.data = self.backup[name]

ema_model = EMA(model, decay=CONFIG["ema_decay"])
ema_embedding = EMA(embedding, decay=CONFIG["ema_decay"])

print(f"✓ EMA enabled with decay={CONFIG['ema_decay']}")

## 5. Training Loop (Corrected)

In [ ]:
import os

# Create checkpoint directory
os.makedirs('/content/drive/MyDrive/TRM', exist_ok=True)

print(f"Starting CORRECTED TRM training for {CONFIG['max_iters']} iterations...")
print("Following paper Algorithm 3:")
print(f"  - n={CONFIG['n_latent']} latent updates per recursion")
print(f"  - T={CONFIG['T_cycles']} cycles (T-1 warmup + 1 grad)")
print(f"  - N_sup={CONFIG['N_sup']} max supervision steps")
print(f"  - 2-layer tiny network")
print(f"  - EMA for stability")
print("="*60)

model.train()
train_iter = iter(dataloader)
start_time = time.time()
best_acc = 0.0

for step in range(CONFIG["max_iters"]):
    try:
        batch = next(train_iter)
    except StopIteration:
        train_iter = iter(dataloader)
        batch = next(train_iter)
    
    x_tokens = batch["x_tokens"].to(device)
    y_true = batch["y_true"].to(device)
    
    # Initialize y with random guesses
    y_tokens = x_tokens.clone()
    blank_mask = y_tokens == 0
    y_tokens[blank_mask] = torch.randint(1, 10, (blank_mask.sum(),), device=device)
    
    # Deep supervision loop (up to N_sup steps)
    total_loss = 0.0
    num_sup_steps = 0
    
    for sup_step in range(CONFIG["N_sup"]):
        optimizer.zero_grad()
        
        with torch.amp.autocast('cuda'):
            x = embedding(x_tokens)
            y = embedding(y_tokens)
            z = torch.zeros_like(x)
            
            # Forward (with T-cycle warmup built-in)
            (y_out, z_out), y_hat, q_hat = model(x, y, z)
            
            # Loss on FINAL output only
            loss = F.cross_entropy(
                y_hat.view(-1, y_hat.size(-1)),
                y_true.view(-1),
                ignore_index=0
            )
        
        # Backprop
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            list(model.parameters()) + list(embedding.parameters()),
            CONFIG["grad_clip"]
        )
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        # Update EMA
        ema_model.update()
        ema_embedding.update()
        
        total_loss += loss.item()
        num_sup_steps += 1
        
        # Update y_tokens for next supervision step
        with torch.no_grad():
            preds = y_hat.argmax(dim=-1)
            y_tokens = x_tokens.clone()
            y_tokens[blank_mask] = preds[blank_mask]
            
            # Early stop if solved
            if (preds[:, :81] == y_true[:, :81]).all():
                break
    
    # Logging
    if step % CONFIG["log_every"] == 0:
        with torch.no_grad():
            # Apply EMA for evaluation
            ema_model.apply_shadow()
            ema_embedding.apply_shadow()
            
            x = ema_embedding(x_tokens)
            y = ema_embedding(batch["y_init_tokens"].to(device))
            z = torch.zeros_like(x)
            (y_out, z_out), y_hat, q_hat = model(x, y, z)
            preds = y_hat[:, :81].argmax(dim=-1)
            acc = (preds == y_true[:, :81]).float().mean().item()
            
            # Restore
            ema_model.restore()
            ema_embedding.restore()
        
        avg_loss = total_loss / num_sup_steps
        elapsed = time.time() - start_time
        lr = scheduler.get_last_lr()[0]
        
        print(f"Step {step:5d}/{CONFIG['max_iters']} | "
              f"Loss: {avg_loss:.4f} | "
              f"Acc: {acc:.3f} | "
              f"SupSteps: {num_sup_steps} | "
              f"LR: {lr:.6f} | "
              f"Time: {elapsed:.0f}s")
        
        if acc > best_acc:
            best_acc = acc
            print(f"  🎯 New best accuracy: {best_acc:.3f}")
    
    # Save checkpoint
    if step > 0 and step % CONFIG["save_every"] == 0:
        ema_model.apply_shadow()
        ema_embedding.apply_shadow()
        
        torch.save({
            "model": model.state_dict(),
            "embedding": embedding.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "step": step,
            "config": CONFIG,
            "best_acc": best_acc
        }, f"/content/drive/MyDrive/TRM/step_{step}.pt")
        
        ema_model.restore()
        ema_embedding.restore()
        
        print(f"  ✓ Saved EMA checkpoint (best_acc: {best_acc:.3f})")

print("="*60)
print(f"Training complete! Best accuracy: {best_acc:.3f}")
print(f"Target: 70-87% (paper: 87.4%)")

In [ ]:
# Save final checkpoint with EMA
ema_model.apply_shadow()
ema_embedding.apply_shadow()

torch.save({
    "model": model.state_dict(),
    "embedding": embedding.state_dict(),
    "step": CONFIG["max_iters"],
    "config": CONFIG,
    "best_acc": best_acc
}, "/content/drive/MyDrive/TRM/final.pt")

print("Saved final.pt with EMA weights to Google Drive!")
print("Download from: /content/drive/MyDrive/TRM/final.pt")

## 6. Quick Evaluation

In [ ]:
# Evaluate on test samples
model.eval()

correct = 0
total = 100

for item in train_data[-100:]:
    puzzle = torch.tensor(item["puzzle"], dtype=torch.long).unsqueeze(0).to(device)
    solution = torch.tensor(item["solution"], dtype=torch.long)
    
    with torch.no_grad():
        x = torch.zeros(1, 128, dtype=torch.long, device=device)
        x[0, :81] = puzzle[0]
        
        y_init = puzzle[0].clone()
        blank_mask = puzzle[0] == 0
        y_init[blank_mask] = torch.randint(1, 10, (blank_mask.sum(),), device=device)
        y = torch.zeros(1, 128, dtype=torch.long, device=device)
        y[0, :81] = y_init
        
        x_emb = embedding(x)
        y_emb = embedding(y)
        z = torch.zeros_like(x_emb)
        
        (_, _), y_hat, _ = model(x_emb, y_emb, z)
        preds = y_hat[0, :81].argmax(dim=-1).cpu()
        
        if (preds == solution).all():
            correct += 1

print(f"Puzzle Accuracy: {correct}/{total} = {correct/total*100:.1f}%")
print(f"Target: 70-87%")
print(f"Status: {'✓ ACHIEVED!' if correct >= 70 else '✗ Continue training' if correct > 50 else '✗ Check training'}")